# Session Data Recovery and Hardware Validation Analysis

**Context:** SD card write failed during this self-calibration 
session. Raw sensor data was recovered from Serial Monitor output 
and reconstructed into a proper timestamped CSV (see 
`reconstruct_session.py` and `trim_session.py` in this repo).

**Goal of this notebook:** Validate that the combined hardware 
(EDA, PPG, accelerometer) produces physiologically plausible 
signals, and investigate whether EDA changes correspond to the 
intended calibration phases (rest, arithmetic, mind-wandering, 
movement).

**Session phases (as intended):**
- Rest baseline
- Mental arithmetic (counting backward by 7s)
- Rest
- Instructed mind-wandering
- Deliberate movement
- Final rest


## 1. Imports and Data Loading

Load the reconstructed session and convert the wall-clock 
timestamp column into elapsed seconds for easier plotting.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import neurokit2 as nk

df = pd.read_csv('session_trimmed.csv')
df['wallclock'] = pd.to_datetime(df['wallclock'])
df['t_sec'] = (df['wallclock'] - df['wallclock'].iloc[0]).dt.total_seconds()

print(f"Session duration: {df['t_sec'].iloc[-1]:.1f} seconds")
print(f"Total samples: {len(df)}")

## 2. Initial Visualization

Plot all three raw signal streams to confirm the hardware is 
producing structured, non-flat, non-saturated data.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

axes[0].plot(df['t_sec'], df['eda_conductance_us'])
axes[0].set_ylabel('EDA (µS)')
axes[0].set_title('Trimmed Session Data')

axes[1].plot(df['t_sec'], df['ppg_ir'])
axes[1].set_ylabel('PPG (IR)')

acc_mag = (df['acc_x']**2 + df['acc_y']**2 + df['acc_z']**2) ** 0.5
axes[2].plot(df['t_sec'], acc_mag)
axes[2].set_ylabel('Accel Magnitude')
axes[2].set_xlabel('Time (seconds)')

plt.tight_layout()
plt.show()

**Finding:** All three signals show real structure -- not flat, 
not saturated. EDA shows a step-like pattern (periods of relative 
stability punctuated by shifts), PPG sits in a sensible range with 
some noisy excursions, and accelerometer is mostly flat with a few 
clear movement bursts. This is the first confirmation that the 
hardware itself is working.

## 3. Signal Quality Check: PPG Dropout

Check for the extended low-PPG dropout identified in the raw 
Serial Monitor data (suspected loss of finger/sensor contact).

In [ ]:
low_ppg = df[df['ppg_ir'] < 5000]
print(f"Rows with suspiciously low PPG: {len(low_ppg)}")
if len(low_ppg) > 0:
    print(f"First occurrence at: {low_ppg['wallclock'].iloc[0]}")
    print(f"Last occurrence at: {low_ppg['wallclock'].iloc[-1]}")

**Finding:** No rows in this trimmed window fall below the 
dropout threshold -- the earlier-identified PPG contact loss occurs 
outside the trimmed session window and does not affect this 
analysis.

## 4. Overlaying Intended Phase Boundaries on EDA

Mark the manually-noted phase start times on the EDA trace to 
check whether physiological changes line up with the intended 
protocol structure.

**Note:** Phase times below were tracked manually by checking a 
clock during the session, not logged automatically -- this is a 
known source of imprecision investigated further below.

In [ ]:
phase_times = {
    'Stabilization end / Rest start': "13:55:00",
    'Arithmetic start': "14:00:00",
    'Rest start': "14:03:00",
    'Mind-wander start': "14:06:00",
    'Movement start': "14:09:00",
    'Final rest start': "14:10:00"
}

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df['t_sec'], df['eda_conductance_us'])
ax.set_ylabel('EDA (µS)')
ax.set_title('EDA with Phase Boundaries')

for label, time_str in phase_times.items():
    phase_time = pd.to_datetime(time_str, format="%H:%M:%S").time()
    matching_rows = df[df['wallclock'].dt.time >= phase_time]
    if len(matching_rows) > 0:
        boundary_t = matching_rows['t_sec'].iloc[0]
        ax.axvline(boundary_t, color='gray', linestyle='--', alpha=0.6)
        ax.text(boundary_t, ax.get_ylim()[1], label,
                rotation=90, fontsize=8, va='top')

plt.tight_layout()
plt.show()

**Finding:** EDA does not rise cleanly during the labeled 
arithmetic phase, and the largest EDA elevation in the session 
occurs mostly *before* the labeled mind-wander phase begins. This 
mismatch is more consistent with phase-timing imprecision than with 
the hardware failing to detect a real cognitive-load response -- 
manually noted clock times carry enough slop (tens of seconds) to 
fully explain this misalignment. **Action item:** use an automated, 
app-driven timer for future calibration sessions instead of manual 
clock-checking.

## 5. Investigating the Largest EDA Spike: Motion Artifact?

The sharpest, largest EDA rise in the session begins right at the 
start of the labeled "movement" phase. Check whether this 
coincides with an accelerometer disturbance.

In [ ]:
acc_mag = (df['acc_x']**2 + df['acc_y']**2 + df['acc_z']**2) ** 0.5

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

axes[0].plot(df['t_sec'], df['eda_conductance_us'])
axes[0].set_ylabel('EDA (µS)')
axes[0].set_title('EDA vs Accelerometer -- Checking for Motion Artifacts')

axes[1].plot(df['t_sec'], acc_mag, color='orange')
axes[1].set_ylabel('Accel Magnitude')
axes[1].set_xlabel('Time (seconds)')

suspicious_windows = [(300, 480), (650, 700), (850, 900)]
for start, end in suspicious_windows:
    for ax in axes:
        ax.axvspan(start, end, color='red', alpha=0.1)

plt.tight_layout()
plt.show()

**Finding:** Only the 850-900s window shows a corresponding 
accelerometer burst -- the 300-480s and 650-700s windows (arithmetic 
and mind-wander, respectively) show flat, quiet accelerometer traces 
despite EDA changes occurring in those windows. This confirms the 
300-480s and 650-700s mismatches are not motion-related (supporting 
the timing-imprecision explanation above), while the 850-900s 
EDA spike does temporally coincide with real movement.

## 6. Is the 850-900s EDA Spike Genuine Arousal or Pure Artifact?

Decompose the movement window into tonic/phasic components and 
compare against a quiet reference window of the same length, to 
test whether the smooth rise-and-decay shape is diagnostic of a 
real physiological response or just an artifact of the tonic 
extraction algorithm.

In [ ]:
movement_mask = (df['t_sec'] >= 840) & (df['t_sec'] <= 900)
movement_eda_raw = df.loc[movement_mask, 'eda_conductance_us'].values

quiet_mask = (df['t_sec'] >= 500) & (df['t_sec'] <= 560)
quiet_eda_raw = df.loc[quiet_mask, 'eda_conductance_us'].values

print(f"Movement window samples: {len(movement_eda_raw)}")
print(f"Quiet window samples: {len(quiet_eda_raw)}")

movement_signals, movement_info = nk.eda_process(
    movement_eda_raw, sampling_rate=4
)
quiet_signals, quiet_info = nk.eda_process(
    quiet_eda_raw, sampling_rate=4
)

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(16, 9), sharex=False)

axes[0, 0].plot(movement_signals['EDA_Raw'], color='tab:blue')
axes[0, 0].set_ylabel('Raw')
axes[0, 0].set_title('Movement Window (840-900s)')

axes[1, 0].plot(movement_signals['EDA_Tonic'], color='tab:blue')
axes[1, 0].set_ylabel('Tonic')

axes[2, 0].plot(movement_signals['EDA_Phasic'], color='tab:blue')
axes[2, 0].set_ylabel('Phasic')
axes[2, 0].set_xlabel('Sample (4 Hz)')

axes[0, 1].plot(quiet_signals['EDA_Raw'], color='tab:orange')
axes[0, 1].set_title('Quiet Window (500-560s)')

axes[1, 1].plot(quiet_signals['EDA_Tonic'], color='tab:orange')

axes[2, 1].plot(quiet_signals['EDA_Phasic'], color='tab:orange')
axes[2, 1].set_xlabel('Sample (4 Hz)')

plt.tight_layout()
plt.show()

In [ ]:
def summarize_window(signals, label):
    tonic = signals['EDA_Tonic'].values
    phasic = signals['EDA_Phasic'].values
    scr_count = signals['SCR_Peaks'].sum()

    print(f"\n--- {label} ---")
    print(f"Tonic range: {tonic.max() - tonic.min():.4f}")
    print(f"Tonic slope (start to peak): "
          f"{tonic.max() - tonic[0]:.4f}")
    print(f"Phasic std: {phasic.std():.4f}")
    print(f"SCR peaks detected: {scr_count}")

    if scr_count > 0:
        peak_amplitudes = signals.loc[
            signals['SCR_Peaks'] == 1, 'SCR_Amplitude'
        ].values
        print(f"Mean SCR amplitude: {peak_amplitudes.mean():.4f}")

summarize_window(movement_signals, "Movement Window")
summarize_window(quiet_signals, "Quiet Window")

**Finding (important -- revises the conclusion above):** The 
quiet window shows a nearly identical smooth bell-shaped tonic 
curve to the movement window, and actually has a *higher* SCR count 
(43 vs 39) and *higher* mean SCR amplitude (0.054 vs 0.046) than the 
movement window. The only metric that favors the movement window is 
tonic range (0.21 vs 0.14), a difference of degree rather than kind.

**Conclusion:** Smooth tonic curve shape is not, on its own, a 
reliable indicator of genuine arousal vs. artifact -- this shape 
appears to be typical of the tonic extraction algorithm's output 
generally, not specific to motion-correlated events. This single 
session does not provide conclusive evidence either way for the 
850-900s spike. The larger tonic range modestly suggests *something* 
different was happening, but this is suggestive, not confirmatory.

**Action item:** disambiguating motion artifact from genuine 
motion-triggered arousal would require either reference-device 
validation (Bland-Altman against research-grade hardware) or a 
much larger systematic comparison across many quiet vs. high-motion 
windows -- not a one-off comparison like this. For the formal study, 
the more practical solution is the conservative exclusion approach 
built below: flag and exclude high-motion windows entirely rather 
than trying to determine whether their EDA content is genuine.

## 7. Hypothesis: Manual Clock-Checking as a Confound

Since phase timing was tracked by periodically glancing at a clock 
rather than via an automated logger, that checking behavior itself 
may have introduced small, repeated attention/motion events 
distinct from the larger, deliberate movement phase.

In [ ]:
zoom_mask = (df['t_sec'] >= 500) & (df['t_sec'] <= 560)

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df.loc[zoom_mask, 't_sec'], acc_mag[zoom_mask])
ax.set_ylabel('Accel Magnitude')
ax.set_xlabel('Time (seconds)')
ax.set_title('Zoomed Accelerometer -- Looking for Small Repeated Blips')
plt.show()

**Finding / open question:** This is a plausible contributing 
factor to the noise observed throughout the session, but was not 
conclusively confirmed or ruled out in this analysis. **Action item: 
for the next session, use an automated app-based timer (or adapt the 
thought-probe app's timer logic) to mark phase transitions instead 
of manually checking a clock.** This removes both the clock-checking 
confound and the manual-timing-imprecision problem identified in 
Section 4.

## 8. Trend-Based View of EDA

As an alternative to analyzing raw EDA values directly, compute a 
smoothed rate-of-change to see whether directional trend reveals 
structure that absolute values obscure.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

axes[0].plot(df['t_sec'], df['eda_conductance_us'])
axes[0].set_ylabel('Raw EDA (µS)')
axes[0].set_title('Raw Values vs. Trend')

rolling_window = 20  # samples, ~5 seconds at 4Hz
eda_smoothed = df['eda_conductance_us'].rolling(
    rolling_window, center=True
).mean()
eda_trend = eda_smoothed.diff()

axes[1].plot(df['t_sec'], eda_trend, color='tab:red')
axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].set_ylabel('Rate of Change')
axes[1].set_xlabel('Time (seconds)')

plt.tight_layout()
plt.show()

**Finding:** Trend view did not resolve the phase-alignment 
ambiguity established above -- it is a useful additional lens but 
does not substitute for fixing phase-boundary timing precision at 
the source.

## 9. Full Pipeline Validation: Windowing, Motion Exclusion, 
Feature Extraction

Run the complete WESAD-style pipeline (sliding window -> motion-based 
artifact exclusion -> NeuroKit2 processing -> feature extraction) 
end-to-end on this session's data, as a smoke test of the full 
analysis pipeline on real device data -- independent of the 
phase-labeling questions above.

In [ ]:
def compute_window_motion(window_df):
    """
    Computes accelerometer magnitude standard deviation
    for a single window. High values indicate motion.
    """
    mag = np.sqrt(
        window_df['acc_x']**2 +
        window_df['acc_y']**2 +
        window_df['acc_z']**2
    )
    return mag.std()


def get_motion_threshold(all_window_motion_values):
    """
    Computes the exclusion threshold for one session,
    based on that session's own distribution of motion values.
    """
    mean_motion = np.mean(all_window_motion_values)
    std_motion = np.std(all_window_motion_values)
    return mean_motion + 2 * std_motion


def extract_features_from_window(window):
    """
    Extract EDA tonic, phasic, and SCR features from a
    single windowed DataFrame segment.

    Parameters
    ----------
    window : pd.DataFrame
        Processed EDA signals for one time window,
        output of nk.eda_process()

    Returns
    -------
    dict : feature name -> value mapping
    """
    features = {}

    tonic = window['EDA_Tonic'].values
    features['tonic_mean'] = np.mean(tonic)
    features['tonic_std'] = np.std(tonic)
    features['tonic_min'] = np.min(tonic)
    features['tonic_max'] = np.max(tonic)
    features['tonic_slope'] = np.polyfit(
        np.arange(len(tonic)), tonic, 1)[0]
    features['tonic_range'] = np.max(tonic) - np.min(tonic)
    features['tonic_cv'] = (
        np.std(tonic) / np.mean(tonic)
        if np.mean(tonic) > 0 else 0
    )

    phasic = window['EDA_Phasic'].values
    features['phasic_mean'] = np.mean(phasic)
    features['phasic_std'] = np.std(phasic)
    features['phasic_min'] = np.min(phasic)
    features['phasic_max'] = np.max(phasic)
    features['phasic_range'] = np.max(phasic) - np.min(phasic)

    scr_peaks = window['SCR_Peaks'].values
    features['scr_count'] = np.sum(scr_peaks)

    peak_amplitudes = window.loc[
        window['SCR_Peaks'] == 1, 'SCR_Amplitude'
    ].values

    features['scr_mean_amplitude'] = (
        np.mean(peak_amplitudes) if len(peak_amplitudes) > 0 else 0
    )
    features['scr_mean_rise_time'] = (
        np.mean(window.loc[
            window['SCR_Peaks'] == 1, 'SCR_RiseTime'
        ].values) if len(peak_amplitudes) > 0 else 0
    )
    features['scr_mean_recovery_time'] = (
        np.mean(window.loc[
            window['SCR_Peaks'] == 1, 'SCR_RecoveryTime'
        ].values) if len(peak_amplitudes) > 0 else 0
    )

    return features

In [ ]:
# Build sliding windows (60s window, 30s step, matching WESAD pipeline)
EDA_RATE = 4  # confirm this matches your actual achieved sampling rate
WINDOW_SIZE = 60
STEP_SIZE = 30

SAMPLES_PER_WINDOW = WINDOW_SIZE * EDA_RATE
SAMPLES_PER_STEP = STEP_SIZE * EDA_RATE

df_reset = df.reset_index(drop=True)

all_windows = []
start = 0
while start + SAMPLES_PER_WINDOW <= len(df_reset):
    end = start + SAMPLES_PER_WINDOW
    all_windows.append(df_reset.iloc[start:end].copy())
    start += SAMPLES_PER_STEP

print(f"Total windows created: {len(all_windows)}")

In [ ]:
# Apply motion-based exclusion
motion_values = [compute_window_motion(w) for w in all_windows]
threshold = get_motion_threshold(motion_values)

clean_windows = []
excluded_count = 0
for window, motion in zip(all_windows, motion_values):
    if motion <= threshold:
        clean_windows.append(window)
    else:
        excluded_count += 1

exclusion_rate = excluded_count / len(all_windows) * 100

print(f"Motion threshold: {threshold:.4f}")
print(f"Windows excluded for motion: {excluded_count}")
print(f"Windows remaining: {len(clean_windows)}")
print(f"Exclusion rate: {exclusion_rate:.1f}%")

In [ ]:
# Process clean windows through NeuroKit2 and extract features
processed_windows = []
for window in clean_windows:
    eda_raw = window['eda_conductance_us'].values
    eda_signals, _ = nk.eda_process(eda_raw, sampling_rate=EDA_RATE)
    processed_windows.append(eda_signals)

clean_features = [
    extract_features_from_window(w) for w in processed_windows
]

feature_df_this_session = pd.DataFrame(clean_features)
print(feature_df_this_session.head())
print(f"\nTotal feature rows: {len(feature_df_this_session)}")

**Finding:** The full pipeline -- windowing, motion-based 
exclusion, NeuroKit2 processing, and feature extraction -- runs 
end-to-end without errors on real device data, producing a feature 
matrix in the same format as the WESAD analysis. Exclusion rate of 
{insert value}% is in a reasonable range given this session 
deliberately included a movement phase. This validates the analysis 
pipeline itself, independent of the phase-labeling questions above.

## 10. Summary and Next Steps

**What this session validated:**
- Hardware (EDA, PPG, ACC) produces real, structured, 
  non-saturated signals
- Full analysis pipeline (windowing -> motion exclusion -> 
  NeuroKit2 -> feature extraction) runs end-to-end on device data
- Accelerometer-based motion exclusion successfully flags the one 
  clearly motion-correlated EDA event in this session

**What this session could not validate:**
- Whether EDA distinguishes cognitive states (arithmetic vs. rest 
  vs. mind-wandering) -- phase boundary timing was not precise 
  enough to test this cleanly
- Whether the large 850-900s EDA spike reflects genuine 
  motion-triggered arousal or measurement artifact -- the 
  tonic/phasic comparison against a quiet window was inconclusive

**Identified causes:**
1. Phase boundaries were tracked manually by checking a clock, 
   introducing timing slop of tens of seconds
2. Manual clock-checking may itself have introduced small, 
   repeated attention/motion events (untested hypothesis)
3. Some sensor connections were not fully secure during this 
   session

**Changes for next session:**
1. Use an automated, app-based timer for phase transitions -- no 
   manual clock-checking
2. Verify all connections are firmly seated before starting; avoid 
   adjusting anything mid-session
3. Minimize incidental movement during non-movement phases
4. Consider adding a longer, explicitly verified quiet baseline 
   period before the first labeled phase
